# Broadcast Variables: Efficient Lookup Distribution via P2P

**Module**: 06_memory_and_storage  
**Notebook**: 06_broadcast_variables.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How broadcast variables work, BitTorrent-like peer-to-peer distribution, and sharing read-only lookup structures across executor tasks.
- **Why does this topic exist?**: Broadcasting copies data to each executor once instead of shipping it with every individual task closure.
- **What problem does it solve?**: Serializing a 50 MB lookup dictionary inside every task saturates driver network.
- **Where is it used in real-world Data Engineering?**: Enriching streaming data with reference tables.

---

## 2. Broadcast Variable Architecture

- Driver serializes object once.
- Distributed to executors via P2P BitTorrent protocol.
- Cached in executor BlockManager for all task threads.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Broadcast_Variables_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Create lookup dictionary on Driver
currency_rates = {"USD": 1.0, "EUR": 0.92, "GBP": 0.79, "EGP": 50.0}

# Broadcast to all worker executors
b_rates = spark.sparkContext.broadcast(currency_rates)
print(f"Broadcast variable ID: {b_rates.id}")
print(f"Broadcast value lookup: USD -> {b_rates.value['USD']}")

# Use inside transformation
sample_rdd = spark.sparkContext.parallelize([("USD", 100), ("EUR", 50), ("EGP", 1000)])
converted = sample_rdd.map(lambda pair: (pair[0], pair[1] / b_rates.value.get(pair[0], 1.0))).collect()
print("Converted amounts:", converted)

---

## Experiment: Unpersisting Broadcast Variable

- **1. What are we testing?**: Call unpersist() and destroy() on broadcast variable.
- **2. What do we expect?**: Releases blocks from executor BlockManagers.

In [ ]:
b_rates.unpersist(); print('Broadcast unpersisted successfully')

**3. What happened?**: Freed executor memory.  
**4. Why did it happen?**: unpersist() removes blocks from workers.  
**5. Practical Engineering Lesson**: Unpersist large broadcast variables when no longer needed.

---

## Practical Exercises

### Exercise 1 (Beginner): Broadcast Lookup Set
**Problem Statement**: Broadcast a set of VIP customer IDs.  
**Input / Context**: Set

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Access in DataFrame with UDF
**Problem Statement**: Use broadcast variable inside a Python UDF.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Broadcast Size Limit
**Problem Statement**: Explain driver memory limits on broadcast.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Broadcast Lookup Set):
```python
b_vips = spark.sparkContext.broadcast({101, 102, 103})
print(101 in b_vips.value)
```

#### Solution 2 (Access in DataFrame with UDF):
```python
@udf(StringType())
def lookup_rate(c):
    return str(b_rates.value.get(c, 1.0))
spark.createDataFrame([('USD',)], ['curr']).select(lookup_rate('curr')).show()
```

#### Solution 3 (Broadcast Size Limit):
```python
print('Broadcast data must fit comfortably in Driver memory and cannot exceed 8GB limit.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Efficiency**: 1 copy per executor JVM, shared by all task threads.
- **Protocol**: BitTorrent-like peer-to-peer distribution.

### Essential Technical Terms
- `Broadcast Variable`: Shared read-only cached object.
- `TorrentBroadcast`: P2P transfer implementation.

### Common Engineering Mistakes
- **Modifying broadcast variable on worker**: Workers cannot mutate broadcast variables; they are strictly read-only.

### Interview Questions on This Topic
- **Q: How does a broadcast variable save memory?**
  - *A*: It copies data once per executor rather than once per task, sharing the memory across all cores on that executor.

### What to Study Next
Open 07_broadcast_join.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')